In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet


import warnings; warnings.simplefilter('ignore')

In [52]:
movies = pd.read_csv('/content/drive/MyDrive/LLM RAG Recommender - MovieLens/Data/ml-32m/movies_enriched.csv')

In [53]:
movies.head()

,movieId,tmdbId,title,overview,tagline,runtime,budget,revenue,release_date,genres,vote_average,vote_count,poster_path,director,cast,keywords,production_companies,collection_name,tmdb_recommendation_ids
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,81,30000000,394436586,1995-11-22,"['Family', 'Comedy', 'Animation', 'Adventure']",7.970,19297,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,John Lasseter,"['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim...","['rescue', 'friendship', 'mission', 'jealousy'...",['Pixar'],Toy Story Collection,"[863, 9487, 10193, 8587, 585]"
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,104,65000000,262821940,1995-12-15,"['Adventure', 'Fantasy', 'Family']",7.242,10971,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,Joe Johnston,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...","['giant insect', 'board game', 'disappearance'...","['TriStar Pictures', 'Interscope Communication...",Jumanji Collection,"[353486, 6795, 788, 1593, 879]"
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,101,25000000,71500000,1995-12-22,"['Romance', 'Comedy']",6.500,409,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,Howard Deutch,"['Walter Matthau', 'Jack Lemmon', 'Ann-Margret...","['fishing', 'sequel', 'old man', 'best friend'...","['Lancaster Gate', 'Warner Bros. Pictures']",Grumpy Old Men Collection,"[11520, 39242, 24732, 30066, 50766]"
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,127,16000000,81452156,1995-12-22,"['Comedy', 'Drama', 'Romance']",6.260,179,/qJU6rfil5xLVb5HpJsmmfeSK254.jpg,Forest Whitaker,"['Whitney Houston', 'Angela Bassett', 'Loretta...","['based on novel or book', 'single mother', 'd...",['20th Century Fox'],NaN,"[31000, 21539, 16158, 33644, 64255]"
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Just when his world is back to normal... he's ...,106,0,76594107,1995-12-08,"['Comedy', 'Family']",6.266,777,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,Charles Shyer,"['Steve Martin', 'Diane Keaton', 'Martin Short...","['daughter', 'baby', 'parent child relationshi...","['Touchstone Pictures', 'Sandollar Productions']",Father of the Bride (Steve Martin) Collection,"[11846, 65137, 22968, 14949, 13759]"


In [54]:
movies.shape

(86315, 19)

In [55]:
import numpy as np
from ast import literal_eval

# replacing nan with empty list []
movies['genres'] = movies['genres'].apply(lambda x: [] if x is np.nan else x)

# converting string into actual list which is the python object
movies['genres'] = movies['genres'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

# Might remove later
movies['genres'] = movies['genres'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])

In [56]:
movies['genres']

,genres
0,"[Family, Comedy, Animation, Adventure]"
1,"[Adventure, Fantasy, Family]"
2,"[Romance, Comedy]"
3,"[Comedy, Drama, Romance]"
4,"[Comedy, Family]"
...,...
86310,[Drama]
86311,"[Drama, Comedy]"
86312,[Drama]
86313,[Drama]


In [57]:
movies.head(2)

,movieId,tmdbId,title,overview,tagline,runtime,budget,revenue,release_date,genres,vote_average,vote_count,poster_path,director,cast,keywords,production_companies,collection_name,tmdb_recommendation_ids
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,81,30000000,394436586,1995-11-22,"[Family, Comedy, Animation, Adventure]",7.970,19297,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,John Lasseter,"['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim...","['rescue', 'friendship', 'mission', 'jealousy'...",['Pixar'],Toy Story Collection,"[863, 9487, 10193, 8587, 585]"
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,104,65000000,262821940,1995-12-15,"[Adventure, Fantasy, Family]",7.242,10971,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,Joe Johnston,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...","['giant insect', 'board game', 'disappearance'...","['TriStar Pictures', 'Interscope Communication...",Jumanji Collection,"[353486, 6795, 788, 1593, 879]"


In [58]:
movies['overview'] = movies['overview'].fillna('')
movies['tagline'] = movies['tagline'].fillna('')
movies['description'] = movies['overview'] + movies['tagline']
movies['title'] = movies['title'].fillna('')
movies['director'] = movies['director'].fillna('')


movies['keywords'] = movies['keywords'].apply(lambda x: [] if x is np.nan else x)
movies['keywords'] = movies['keywords'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
movies['keywords'] = movies['keywords'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])

movies['cast'] = movies['cast'].apply(lambda x: [] if x is np.nan else x)
movies['cast'] = movies['cast'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
movies['cast'] = movies['cast'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])
movies['cast'] = movies['cast'].apply(lambda x: x[:3] if len(x) >=3 else x)

movies['production_companies'] = movies['production_companies'].apply(lambda x: [] if x is np.nan else x)
movies['production_companies'] = movies['production_companies'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
movies['production_companies'] = movies['production_companies'].apply(lambda x: [i for i in x] if isinstance(x, list) else [])
movies['production_companies'] = movies['production_companies'].apply(lambda x: x[:3] if len(x) >=3 else x)


In [59]:
movies.cast, movies.production_companies

(0                      [Tom Hanks, Tim Allen, Don Rickles]
 1          [Robin Williams, Kirsten Dunst, Bradley Pierce]
 2               [Walter Matthau, Jack Lemmon, Ann-Margret]
 3        [Whitney Houston, Angela Bassett, Loretta Devine]
 4               [Steve Martin, Diane Keaton, Martin Short]
                                ...                        
 86310          [Damián Alcázar, Grapa Paola, María Zubiri]
 86311    [Siobhan Fallon Hogan, Peter Macon, Robert Pat...
 86312    [Taraneh Alidoosti, Mahtab Keramati, Masoud Ka...
 86313      [Jan Sterling, James MacArthur, William Windom]
 86314                            [Ueli Steck, Dani Arnold]
 Name: cast, Length: 86315, dtype: object,
 0                                                  [Pixar]
 1        [TriStar Pictures, Interscope Communications, ...
 2                  [Lancaster Gate, Warner Bros. Pictures]
 3                                       [20th Century Fox]
 4             [Touchstone Pictures, Sandollar Production

In [60]:
movies.description

,description
0,"Led by Woody, Andy's toys live happily in his ..."
1,When siblings Judy and Peter discover an encha...
2,A family wedding reignites the ancient feud be...
3,"Cheated on, mistreated and stepped on, the wom..."
4,Just when George Banks has recovered from his ...
...,...
86310,Ronnie Monroy has had an unmeaningful life as ...
86311,A death row prisoner with 10 days left to live...
86312,"Elham is a young, divorced Iranian woman. Seek..."
86313,"In Vietnam, aspiring actor Johnny Taylor is gi..."


In [61]:
movies['cast'] = movies['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])
movies['director'] = movies['director'].apply(lambda x: str.lower(x.replace(" ", "")))


In [62]:
def weighted_rating(x, m, C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

In [63]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1)

In [64]:
tfidf_matrix = tfidf.fit_transform(movies['description'])

In [65]:
tfidf_matrix.shape

(86315, 1736558)

In [66]:
from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer

In [67]:
def create_soup(x):
    return (
        ' '.join(x['keywords']) + ' ' +
        ' '.join(x['cast']) + ' ' +
        x['director'] + ' ' +
        ' '.join(x['genres']) + ' ' +
        ' '.join(x['production_companies'])
    )

movies['soup'] = movies.apply(create_soup, axis=1)

In [68]:
count = CountVectorizer(analyzer='word', ngram_range=(1, 1), min_df=1, stop_words='english')
count_matrix = count.fit_transform(movies['soup'])

In [69]:
count_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1332414 stored elements and shape (86315, 162458)>

In [70]:
from scipy.sparse import hstack

# Stack the two matrices horizontally
combined_matrix = hstack([tfidf_matrix, count_matrix])

In [71]:
combined_matrix.shape

(86315, 1899016)

In [72]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


movies = movies.reset_index()
indices = pd.Series(movies.index, index=movies['title'])
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year


In [73]:
def get_recommendations_batch(titles, combined_matrix, movies_df, indices, batch_size=10, top_n=10, candidate_pool=50):

    results = {}

    # Process in batches
    for i in range(0, len(titles), batch_size):
        batch_titles = titles[i : i + batch_size]

        # Get valid indices
        batch_indices = []
        valid_titles = []

        for t in batch_titles:
            if t in indices:
                idx = indices[t]
                # Handle duplicate titles
                if isinstance(idx, (pd.Series, pd.Index, np.ndarray, list)):
                    idx = idx.iloc[0] if hasattr(idx, 'iloc') else idx[0]

                batch_indices.append(idx)
                valid_titles.append(t)
            else:
                print(f"Warning: '{t}' not found in indices.")

        if not batch_indices:
            continue

        # Extract feature vectors
        target_vectors = combined_matrix[batch_indices]

        # Compute cosine similarity
        batch_sim_scores = cosine_similarity(target_vectors, combined_matrix)

        # Process each movie
        for k, title in enumerate(valid_titles):
            scores = batch_sim_scores[k]

            top_indices = np.argpartition(scores, -(candidate_pool + 1))[-(candidate_pool + 1):]
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

            # Exclude input movie
            final_indices = [idx for idx in top_indices if idx != batch_indices[k]][:candidate_pool]

            # Get metadata
            recs = movies_df.iloc[final_indices][['title', 'vote_count', 'vote_average', 'year']].copy()
            recs['similarity'] = scores[final_indices]

            # Filter for Quality

            C = recs['vote_average'].mean()
            m = recs['vote_count'].quantile(0.20)
            recs = recs[recs['vote_count'] >= m]

            # Calculate Score
            recs['wr_score'] = recs.apply(lambda x: weighted_rating(x, m, C), axis=1)


            results[title] = recs.sort_values('similarity', ascending=False).head(top_n)

    return results

In [74]:
# import time


# test_movies = [
#     'Toy Story', 'Jumanji', 'Grumpier Old Men', 'Heat', 'Sabrina', 'Tom and Huck',
#     'Sudden Death', 'GoldenEye', 'The American President', 'Dracula: Dead and Loving It', 'The Batman'
# ]

# start = time.time()
# recs_1 = get_recommendations_batch(test_movies, combined_matrix, movies, indices, batch_size=1)
# end = time.time()
# print(f"Batch Size 1 took: {end - start:.4f} seconds")


# recs_10 = get_recommendations_batch(test_movies, combined_matrix, movies, indices, batch_size=1000)



# print(recs_10['The Batman'])

In [75]:
import time
import random

# Selecting 100 Random Movies
random_titles = movies['title'].sample(n=100, random_state=42).tolist()

print(f"Selected {len(random_titles)} random movies for testing.")
print(f"Examples: {random_titles[:5]}")




test_results = get_recommendations_batch(
    random_titles,
    combined_matrix,
    movies,
    indices,
    batch_size=1000,
    top_n=50,
    candidate_pool=100
)


# looking at random results to see if they make sense
for i in range(10):
    title = random_titles[i]
    print(f"\nSource Movie: {title}")
    if title in test_results:
        print(test_results[title][['title', 'similarity', 'wr_score']].head(100))
    else:
        print("No recommendations generated (possibly filtered out).")

Selected 100 random movies for testing.
Examples: ['The Penthouse', 'Jinn', 'Robot on the Road', 'Pigs 2: The Last Blood', 'Crooklyn']

Source Movie: The Penthouse
                            title  similarity  wr_score
52300            The Lost Brother    0.410391  6.778114
51611             Dead Slow Ahead    0.389490  6.482203
51579                 Mrs. Harris    0.382692  5.704024
79083            The Night Doctor    0.381771  6.445280
23937             Penthouse North    0.377815  5.710899
48371               The Next Skin    0.374634  5.696419
23614                  On the Ice    0.374634  6.108066
23992        The Body of My Enemy    0.368606  6.324577
39647         A Friend of Vincent    0.368563  5.787305
24059                       Prowl    0.368421  5.030127
83654                    Scorpion    0.368421  5.446472
67927                  Dirt Music    0.368421  6.138481
68112        The Crimes That Bind    0.367884  6.397501
15068              8th Wonderland    0.367884  6.051

In [76]:
#Trying the weighted approach

In [77]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. Define Vectorizers
count_vec = CountVectorizer(stop_words='english', min_df=1)
tfidf_vec = TfidfVectorizer(stop_words='english', min_df=1)

# 2. Fit Separate Matrices
# We assume your 'movies' dataframe already has these columns cleaned
print("Building separate matrices...")

# Matrix A: Description (Plot)
mat_desc = tfidf_vec.fit_transform(movies['description'])

# Matrix B: Keywords (Plot elements)
# (If keywords are lists, join them first: movies['keywords'].apply(lambda x: ' '.join(x)))
mat_keywords = count_vec.fit_transform(movies['keywords'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

# Matrix C: Director (Style/Vibe)
mat_director = count_vec.fit_transform(movies['director'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

# Matrix D: Genres (Category)
mat_genres = count_vec.fit_transform(movies['genres'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

# Matrix E: Cast (Star Power)
mat_cast = count_vec.fit_transform(movies['cast'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

mat_production = count_vec.fit_transform(movies['production_companies'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x))

mat_title = tfidf_vec.fit_transform(movies['title'])

Building separate matrices...


In [78]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

def get_weighted_recommendations(title, weights, movies_df, indices, top_n=20):
    """
    weights: Dictionary of weights, e.g., {'desc': 0.1, 'director': 0.5, ...}
    """
    if title not in indices:
        return "Movie not found."

    idx = indices[title]

    # Handle duplicate titles
    if isinstance(idx, (pd.Series, pd.Index, np.ndarray, list)):
        idx = idx.iloc[0]


    # Description
    sim_desc = cosine_similarity(mat_desc[idx], mat_desc).flatten()

    # Keywords
    sim_keywords = cosine_similarity(mat_keywords[idx], mat_keywords).flatten()

    # Director (Crucial for vibe!)
    sim_director = cosine_similarity(mat_director[idx], mat_director).flatten()

    # Genres
    sim_genres = cosine_similarity(mat_genres[idx], mat_genres).flatten()

    # Cast
    sim_cast = cosine_similarity(mat_cast[idx], mat_cast).flatten()

    sim_production = cosine_similarity(mat_production[idx], mat_production).flatten()

    sim_title = cosine_similarity(mat_title[idx], mat_title).flatten()
    # Calculating Weighted Average Score

    final_score = (
        (sim_desc * weights.get('desc', 0)) +
        (sim_keywords * weights.get('keywords', 0)) +
        (sim_director * weights.get('director', 0)) +
        (sim_genres * weights.get('genres', 0)) +
        (sim_cast * weights.get('cast', 0)) +
        (sim_production * weights.get('production', 0)) +
        (sim_title * weights.get('title', 0))
    )


    # Get top candidates
    candidate_pool = 100
    top_indices = np.argpartition(final_score, -(candidate_pool + 1))[-(candidate_pool + 1):]
    top_indices = top_indices[np.argsort(final_score[top_indices])[::-1]]

    # Excluding the movie itself
    final_indices = [i for i in top_indices if i != idx]

    # Metadata
    recs = movies_df.iloc[final_indices][['title', 'vote_count', 'vote_average', 'year']].copy()


    recs['sim_total'] = final_score[final_indices]
    recs['sim_director'] = sim_director[final_indices]
    recs['sim_desc'] = sim_desc[final_indices]
    recs['sim_keywords'] = sim_keywords[final_indices]
    recs['sim_production'] = sim_production[final_indices]
    recs['sim_title'] = sim_title[final_indices]

    # Weighted Rating Sort
    C = recs['vote_average'].mean()
    m = recs['vote_count'].quantile(0.30)
    recs = recs[recs['vote_count'] >= m]
    recs['wr_score'] = recs.apply(lambda x: weighted_rating(x, m, C), axis=1)

    return recs.sort_values('sim_total', ascending=False).head(top_n)

In [79]:
import pandas as pd


custom_weights = {
    'desc': 0.2,
    'keywords': 0.4,
    'director': 0.5,
    'genres': 0.4,
    'production': 0.2,
    'cast': 0.2,
    'title': 0.3
}

results = get_weighted_recommendations("The Dark Knight Rises", custom_weights, movies, indices)
print(results[['title', 'sim_total', 'sim_director', 'sim_desc', 'wr_score', 'sim_keywords', 'sim_production', 'sim_title']])

                                 title  sim_total  sim_director  sim_desc  \
12165                  The Dark Knight   1.535447           1.0  0.325888   
9960                     Batman Begins   1.214011           1.0  0.202808   
5258                          Insomnia   0.896266           1.0  0.000000   
47062                          Dunkirk   0.876054           1.0  0.000000   
2471                         Following   0.840653           1.0  0.030851   
66470                            Tenet   0.828835           1.0  0.000000   
84910                      Oppenheimer   0.811961           1.0  0.000000   
20984                     Interstellar   0.745056           1.0  0.000000   
11115                     The Prestige   0.740752           1.0  0.013564   
14844                        Inception   0.740284           1.0  0.000000   
4102                           Memento   0.688441           1.0  0.000000   
4510                      The Punisher   0.574132           0.0  0.011972   

In [80]:
test_movies = movies['title'].sample(n=100, random_state=42).tolist()

for movie in test_movies:
    results = get_weighted_recommendations(movie, custom_weights, movies, indices)
    print(f"\nSource Movie: {movie}")
    print(results[['title', 'sim_total', 'sim_director', 'sim_desc', 'wr_score', 'sim_keywords', 'sim_production', 'sim_title']])

# results = get_weighted_recommendations("The Dark Knight Rises", custom_weights, movies, indices)
# print(results[['title', 'sim_total', 'sim_director', 'sim_desc', 'wr_score', 'sim_keywords', 'sim_production', 'sim_title']])


Source Movie: The Penthouse
                           title  sim_total  sim_director  sim_desc  wr_score  \
23937            Penthouse North   0.874865           0.0  0.242440  5.700393   
17233                     Fright   0.859290           1.0  0.013856  5.918764   
28994                Open Season   0.826599           1.0  0.000000  5.368057   
29048       The Long Day's Dying   0.766667           1.0  0.000000  5.356483   
20890   And Then There Were None   0.703143           1.0  0.000000  5.661761   
23466   Straight On Till Morning   0.701808           1.0  0.009042  5.624405   
4806               The Earthling   0.667466           1.0  0.020834  5.981774   
74159              The Penthouse   0.662801           0.0  0.199790  4.287146   
6746             The Italian Job   0.641421           1.0  0.000000  6.956345   
12125                    Descent   0.621320           0.0  0.000000  4.463193   
4838   The Business of Strangers   0.592985           0.0  0.012090  5.510064   

# Dynamic Weight system

In [81]:
# STEP 1: BUILD INVERTED INDICES (LOOKUP DICTIONARIES)

from collections import defaultdict
import pandas as pd
import numpy as np

def normalize_name(name):
    """
    Normalize names to handle multiple formats.
    'Christopher Nolan' -> 'christophernolan'
    'christopher nolan' -> 'christophernolan'
    'christophernolan' -> 'christophernolan'
    """
    if pd.isna(name) or not str(name).strip():
        return None
    return re.sub(r'\s+', '', str(name).lower().strip())

def build_lookup_indices(movies_df):
    """
    Build hash maps for fast categorical lookups.
    Returns dict of dicts: entity_type -> {entity_value: [movie_indices]}
    """

    # 1. Director Lookup
    director_lookup = defaultdict(list)
    for idx, director in enumerate(movies_df['director']):
        if pd.notna(director) and str(director).strip():
            # Store both original and lowercase for flexible matching
            director_lookup[str(director).lower().strip()].append(idx)

    # 2. Genre Lookup
    genre_lookup = defaultdict(list)
    for idx, genres in enumerate(movies_df['genres']):
        if isinstance(genres, list):
            for genre in genres:
                genre_lookup[str(genre).lower().strip()].append(idx)
        elif isinstance(genres, str) and genres:
            # Handle comma-separated string format
            for genre in genres.split(','):
                genre_lookup[genre.lower().strip()].append(idx)

    # 3. Keyword Lookup
    keyword_lookup = defaultdict(list)
    for idx, keywords in enumerate(movies_df['keywords']):
        if isinstance(keywords, list):
            for kw in keywords:
                keyword_lookup[str(kw).lower().strip()].append(idx)
        elif isinstance(keywords, str) and keywords:
            for kw in keywords.split(','):
                keyword_lookup[kw.lower().strip()].append(idx)

    # 4. Cast Lookup (top 5 actors per movie)
    cast_lookup = defaultdict(list)
    for idx, cast_list in enumerate(movies_df['cast']):
        if isinstance(cast_list, list):
            for actor in cast_list[:5]:
                cast_lookup[str(actor).lower().strip()].append(idx)
        elif isinstance(cast_list, str) and cast_list:
            for actor in cast_list.split(',')[:3]:
                cast_lookup[actor.lower().strip()].append(idx)

    # 5. Production Company Lookup
    production_lookup = defaultdict(list)
    for idx, companies in enumerate(movies_df['production_companies']):
        if isinstance(companies, list):
            for company in companies[:3]:
                production_lookup[str(company).lower().strip()].append(idx)
        elif isinstance(companies, str) and companies:
            for company in companies.split(',')[:1]:
                production_lookup[company.lower().strip()].append(idx)

    return {
        'director': dict(director_lookup),
        'genre': dict(genre_lookup),
        'keyword': dict(keyword_lookup),
        'cast': dict(cast_lookup),
        'production': dict(production_lookup)
    }

# BUILD THE LOOKUPS (run once)
lookups = build_lookup_indices(movies)

print("=== Lookup Statistics ===")
print(f"Unique Directors: {len(lookups['director']):,}")
print(f"Unique Genres: {len(lookups['genre']):,}")
print(f"Unique Keywords: {len(lookups['keyword']):,}")
print(f"Unique Actors: {len(lookups['cast']):,}")
print(f"Unique Production Companies: {len(lookups['production']):,}")

=== Lookup Statistics ===
Unique Directors: 34,175
Unique Genres: 19
Unique Keywords: 30,591
Unique Actors: 92,494
Unique Production Companies: 42,404


In [95]:
# STEP 1: BUILD OPTIMIZED INVERTED INDICES (LOOKUP DICTIONARIES)

import pandas as pd
import numpy as np
import re
from collections import defaultdict

def sanitize_key(text):
    """
    Converts input into a standardized 'soup' key.
    Examples:
    'Science Fiction' -> 'sciencefiction'
    'Christopher Nolan' -> 'christophernolan'
    '  Action ' -> 'action'
    """
    if pd.isna(text) or text == "":
        return None

    # Convert to string, lowercase, and remove ALL whitespace (not just trim)
    clean_text = str(text).lower()
    clean_text = re.sub(r'\s+', '', clean_text)

    return clean_text if clean_text else None

def build_lookup_indices(movies_df):
    """
    Builds hash maps mapping 'sanitized_key' -> [List of Movie Indices].
    This allows O(1) retrieval without scanning columns.
    """

    # Initialize dictionaries
    lookups = {
        'director': defaultdict(list),
        'genre': defaultdict(list),
        'keyword': defaultdict(list),
        'cast': defaultdict(list),
        'production': defaultdict(list)
    }

    # Iterate over the dataframe once (More efficient than column-wise processing)
    for idx, row in movies_df.iterrows():

        # 1. Director Lookup
        d_key = sanitize_key(row['director'])
        if d_key:
            lookups['director'][d_key].append(idx)

        # 2. Genre Lookup (Handle lists or strings)
        genres = row['genres']
        if isinstance(genres, list):
            for g in genres:
                g_key = sanitize_key(g)
                if g_key: lookups['genre'][g_key].append(idx)
        elif isinstance(genres, str):
            # Fallback for string representations like "Action, Comedy"
            for g in genres.split(','):
                g_key = sanitize_key(g)
                if g_key: lookups['genre'][g_key].append(idx)

        # 3. Keyword Lookup
        keywords = row['keywords']
        if isinstance(keywords, list):
            for k in keywords:
                k_key = sanitize_key(k)
                if k_key: lookups['keyword'][k_key].append(idx)

        # 4. Cast Lookup (Limit to top 5 to reduce noise)
        cast = row['cast']
        if isinstance(cast, list):
            for actor in cast[:5]:
                a_key = sanitize_key(actor)
                if a_key: lookups['cast'][a_key].append(idx)

        # 5. Production Company Lookup (Limit to top 3)
        companies = row['production_companies']
        if isinstance(companies, list):
            for c in companies[:3]:
                c_key = sanitize_key(c)
                if c_key: lookups['production'][c_key].append(idx)

    # Convert defaultdicts to standard dicts for cleaner printing/usage
    return {k: dict(v) for k, v in lookups.items()}

# ---------------------------------------------------------
# EXECUTE BUILD
# ---------------------------------------------------------
print("Building Inverted Indices...")
lookups = build_lookup_indices(movies)

print("\n=== Lookup Statistics (Nodes in Graph) ===")
print(f"Unique Directors:  {len(lookups['director']):,}")
print(f"Unique Genres:     {len(lookups['genre']):,}")
print(f"Unique Keywords:   {len(lookups['keyword']):,}")
print(f"Unique Actors:     {len(lookups['cast']):,}")
print(f"Unique Companies:  {len(lookups['production']):,}")



Building Inverted Indices...

=== Lookup Statistics (Nodes in Graph) ===
Unique Directors:  34,175
Unique Genres:     19
Unique Keywords:   30,408
Unique Actors:     92,494
Unique Companies:  42,248

[Verification] Found 5395 movies for 'sciencefiction'


In [96]:
# Verification Test
test_director = "sciencefiction"
if test_director in lookups['genre']:
    print(f"\n[Verification] Found {len(lookups['genre'][test_director])} movies for '{test_director}'")
else:
    print(f"\n[Verification] '{test_director}' not found (Check sanitization logic)")


[Verification] Found 5395 movies for 'sciencefiction'


In [97]:
# DYNAMIC WEIGHT CALCULATOR

def calculate_dynamic_weights(movie=None, director=None, genres=None,
                               keywords=None, actor=None):
    """
    Dynamically calculate weights based on provided inputs.

    Logic:
    - More inputs = distribute weights across them
    - If seed movie provided = movie similarity gets higher base weight
    - If only attributes (no movie) = attributes share the weight pool
    """

    # Count active inputs
    active_inputs = []
    if movie: active_inputs.append('movie')
    if director: active_inputs.append('director')
    if genres: active_inputs.append('genre')
    if keywords: active_inputs.append('keyword')
    if actor: active_inputs.append('actor')

    num_active = len(active_inputs)

    if num_active == 0:
        return None, "No inputs provided"

        # {This should give the current popular movies instead of nothing}

    # Base weight distribution
    weights = {
        'movie': 0.5,
        'director': 0.7,
        'genre': 0.6,
        'keyword': 0.6,
        'actor': 0.1
    }

    # WEIGHT ASSIGNMENT LOGIC

    if num_active == 1:
        # Single input gets full weight
        weights[active_inputs[0]] = 1.0

    elif movie and num_active == 2:
        # Movie + 1 attribute: Movie dominant
        weights['movie'] = 0.6
        other = [x for x in active_inputs if x != 'movie'][0]
        weights[other] = 0.4

    elif movie and num_active >= 3:
        # Movie + multiple attributes: Movie still important but shared
        weights['movie'] = 0.4
        remaining = 0.6
        others = [x for x in active_inputs if x != 'movie']
        per_other = remaining / len(others)
        for o in others:
            weights[o] = per_other

    else:
        # No movie (pure attribute search): Equal distribution with genre boost
        base_weight = 1.0 / num_active
        for inp in active_inputs:
            weights[inp] = base_weight
        # Slightly boost director if present
        if 'director' in active_inputs:
            weights['director'] *= 1.3
            # Renormalize
            total = sum(weights.values())
            weights = {k: v/total for k, v in weights.items()}

    return weights, active_inputs

# Test the weight calculator
test_cases = [
    {'movie': 'Interstellar'},
    {'movie': 'Interstellar', 'director': 'christophernolan'},
    {'director': 'christophernolan', 'genres': ['Thriller']},
    {'movie': 'Inception', 'genres': ['Sci-Fi'], 'keywords': ['dream', 'heist']},
]

print("=== Dynamic Weight Tests ===")
for tc in test_cases:
    w, active = calculate_dynamic_weights(**tc)
    print(f"Input: {tc}")
    print(f"  Weights: {w}")
    print()


=== Dynamic Weight Tests ===
Input: {'movie': 'Interstellar'}
  Weights: {'movie': 1.0, 'director': 0.7, 'genre': 0.6, 'keyword': 0.6, 'actor': 0.1}

Input: {'movie': 'Interstellar', 'director': 'christophernolan'}
  Weights: {'movie': 0.6, 'director': 0.4, 'genre': 0.6, 'keyword': 0.6, 'actor': 0.1}

Input: {'director': 'christophernolan', 'genres': ['Thriller']}
  Weights: {'movie': 0.2127659574468085, 'director': 0.2765957446808511, 'genre': 0.2127659574468085, 'keyword': 0.2553191489361702, 'actor': 0.0425531914893617}

Input: {'movie': 'Inception', 'genres': ['Sci-Fi'], 'keywords': ['dream', 'heist']}
  Weights: {'movie': 0.4, 'director': 0.7, 'genre': 0.3, 'keyword': 0.3, 'actor': 0.1}



In [98]:
# UNIFIED SCORING ENGINE

from sklearn.metrics.pairwise import cosine_similarity

def compute_dynamic_scores(
    movies_df,
    indices,
    mat_desc,
    lookups,
    movie=None,
    director=None,
    genres=None,
    keywords=None,
    actor=None,
    custom_weights=None
):
    """
    Compute scores for all movies based on provided inputs.

    Returns:
        np.array of scores for each movie (shape: num_movies,)
    """

    num_movies = len(movies_df)
    total_scores = np.zeros(num_movies)

    # Get weights
    if custom_weights:
        weights = custom_weights
    else:
        weights, active_inputs = calculate_dynamic_weights(
            movie=movie, director=director, genres=genres,
            keywords=keywords, actor=actor
        )
        if weights is None:
            return total_scores

    seed_idx = None

    # MOVIE SIMILARITY
    if movie and weights.get('movie', 0) > 0:
        if movie in indices:
            seed_idx = indices[movie]

            # Handle duplicate titles
            if isinstance(seed_idx, (pd.Series, pd.Index, np.ndarray, list)):
                seed_idx = seed_idx.iloc[0] if hasattr(seed_idx, 'iloc') else seed_idx[0]

            # Cosine similarity against all movies
            sim_scores = cosine_similarity(
                mat_desc[seed_idx],
                mat_desc
            ).flatten()

            total_scores += sim_scores * weights['movie']
        else:
            print(f"Warning: Movie '{movie}' not found in index")

    # DIRECTOR BOOST (Sparse)
    if director and weights.get('director', 0) > 0:
        director_key = director.lower().strip()
        director_key = normalize_name(director_key)
        target_indices = lookups['director'].get(director_key, [])

        if target_indices:
            # Boost all movies by this director
            total_scores[target_indices] += weights['director']
        else:
            print(f"Warning: Director '{director}' not found in lookup")

    # GENRE BOOST (Sparse)
    if genres and weights.get('genre', 0) > 0:
        if isinstance(genres, str):
            genres = [genres]

        genre_weight_per = weights['genre'] / len(genres)

        for genre in genres:
            genre_key = genre.lower().strip()
            genre_key = normalize_name(genre_key)
            target_indices = lookups['genre'].get(genre_key, [])

            if target_indices:
                total_scores[target_indices] += genre_weight_per

    # KEYWORD BOOST (Sparse)
    if keywords and weights.get('keyword', 0) > 0:
        if isinstance(keywords, str):
            keywords = [keywords]

        keyword_weight_per = weights['keyword'] / len(keywords)

        for kw in keywords:
            kw_key = kw.lower().strip()
            kw_key = normalize_name(kw_key)
            target_indices = lookups['keyword'].get(kw_key, [])

            if target_indices:
                total_scores[target_indices] += keyword_weight_per

    # ACTOR BOOST (Sparse)
    if actor and weights.get('actor', 0) > 0:
        actor_key = actor.lower().strip()
        actor_key = normalize_name(actor_key)
        target_indices = lookups['cast'].get(actor_key, [])

        if target_indices:
            total_scores[target_indices] += weights['actor']

    return total_scores, seed_idx, weights


In [99]:
# MAIN RECOMMENDATION FUNCTION

def get_dynamic_recommendations(
    movies_df,
    indices,
    mat_desc,
    lookups,
    movie=None,
    director=None,
    genres=None,
    keywords=None,
    actor=None,
    custom_weights=None,
    top_n=15,
    min_votes=None,
    verbose=True
):
    """
    Get movie recommendations based on any combination of inputs.

    Parameters:
    -----------
    movies_df : pd.DataFrame
        Your movies dataframe
    indices : pd.Series
        Title to index mapping
    mat_desc : sparse matrix
        TF-IDF matrix for descriptions
    lookups : dict
        Inverted indices from build_lookup_indices()
    movie : str, optional
        Seed movie title for similarity
    director : str, optional
        Director name to boost
    genres : list or str, optional
        Genre(s) to boost
    keywords : list or str, optional
        Keyword(s) to boost
    actor : str, optional
        Actor name to boost
    custom_weights : dict, optional
        Override automatic weights
    top_n : int
        Number of recommendations to return
    min_votes : int, optional
        Minimum vote count filter
    verbose : bool
        Print debug information

    Returns:
    --------
    pd.DataFrame with recommendations
    """

    # Compute scores
    scores, seed_idx, weights = compute_dynamic_scores(
        movies_df=movies_df,
        indices=indices,
        mat_desc=mat_desc,
        lookups=lookups,
        movie=movie,
        director=director,
        genres=genres,
        keywords=keywords,
        actor=actor,
        custom_weights=custom_weights
    )

    if verbose:
        print(f"Inputs:")
        if movie: print(f"  Movie: {movie}")
        if director: print(f"  Director: {director}")
        if genres: print(f"  Genres: {genres}")
        if keywords: print(f"  Keywords: {keywords}")
        if actor: print(f"  Actor: {actor}")
        print(f"\nCalculated Weights:")
        for k, v in weights.items():
            if v > 0:
                print(f"  {k}: {v:.3f}")
        print("-" * 60)

    # Get top candidates (more than needed for filtering)
    candidate_pool = min(top_n * 3, len(movies_df))
    top_indices = np.argsort(scores)[::-1][:candidate_pool]

    # Build results DataFrame
    result_cols = ['title', 'director', 'genres', 'keywords',
                   'voteaverage', 'votecount', 'year']
    available_cols = [c for c in result_cols if c in movies_df.columns]

    results = movies_df.iloc[top_indices][available_cols].copy()
    results['score'] = scores[top_indices]

    # Exclude seed movie if provided
    if seed_idx is not None:
        results = results[results.index != seed_idx]

    # Apply minimum votes filter
    if min_votes and 'votecount' in results.columns:
        results = results[results['votecount'] >= min_votes]

    # Calculate weighted rating (your existing formula)
    if 'voteaverage' in results.columns and 'votecount' in results.columns:
        C = results['voteaverage'].mean()
        m = results['votecount'].quantile(0.25)

        def weighted_rating(x, m=m, C=C):
            v = x['votecount']
            R = x['voteaverage']
            return (v / (v + m) * R) + (m / (v + m) * C)

        results['wr_score'] = results.apply(weighted_rating, axis=1)

    results = results.head(top_n)

    if verbose:
        print(f"Returning {len(results)} recommendations")

    return results


In [100]:
# TEST THE SYSTEM

print("Movie Only (like your original system)")
recs1 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    movie='Interstellar',
    top_n=10
)
print(recs1[['title', 'director', 'score']].to_string())


print("TEST 2: Movie + Director")
print("="*70)
recs2 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    movie='Interstellar',
    director='Christopher Nolan',
    top_n=10
)
print(recs2[['title', 'director', 'score']].to_string())


print("TEST 3: Movie + Genre (Interstellar but Thriller)")
print("="*70)
recs3 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    movie='Interstellar',
    genres=['Thriller'],
    top_n=10
)
print(recs3[['title', 'genres', 'score']].to_string())


print("TEST 4: Director + Genre Only (No Seed Movie)")
recs4 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    director='David Fincher',
    genres=['Thriller'],
    top_n=10
)
print(recs4[['title', 'director', 'genres', 'score']].to_string())


print("TEST 5: Movie + Multiple Attributes")
recs5 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    movie='The Dark Knight',
    director='Christopher Nolan',
    genres=['Action', 'Crime'],
    keywords=['superhero', 'vigilante'],
    top_n=10
)
print(recs5[['title', 'director', 'score']].to_string())


print("TEST 6: Keywords Only (Concept Search)")
recs6 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    keywords=['space', 'astronaut', 'survival'],
    top_n=10
)
print(recs6[['title', 'keywords', 'score']].to_string())


print("TEST 7: Custom Weights Override")
# custom = {
#     'movie': 0.1,
#     'director': 0.1,
#     'genre': 0.1,
#     'keyword': 0.1,
#     'actor': 0.0
# }
recs7 = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    movie='Inception',
    director='David Fincher',
    genres=['Science Fiction'],
    top_n=10
)
print(recs7[['title', 'director', 'score']].to_string())


Movie Only (like your original system)
Inputs:
  Movie: Interstellar

Calculated Weights:
  movie: 1.000
  director: 0.700
  genre: 0.600
  keyword: 0.600
  actor: 0.100
------------------------------------------------------------
Returning 10 recommendations
                      title           director     score
51230            The Beyond       hasrafdulull  0.225105
18003            Prometheus        ridleyscott  0.174830
80155                Origin      davidparrella  0.168225
75889                Within  olatundeosunsanmi  0.165968
54283     A Species Odyssey   jacquesmalaterre  0.155566
80011         The Prototype       marcelogrion  0.151959
37366  AE: Apocalypse Earth       thunderlevin  0.151832
42678     Voyage to the Sky       jeanpainlevé  0.150714
47766           Anti Matter        keirburrows  0.144483
59010               Genesis        osamutezuka  0.143837
TEST 2: Movie + Director
Inputs:
  Movie: Interstellar
  Director: Christopher Nolan

Calculated Weights:
  movie

In [102]:
# MINIMAL SETUP

# 1. Build lookups (run once after data loading)
lookups = build_lookup_indices(movies)

# 2. Get recommendations (call anytime)
recommendations = get_dynamic_recommendations(
    movies_df=movies,
    indices=indices,
    mat_desc=mat_desc,
    lookups=lookups,
    movie='Interstellar',
    director='Christopher Nolan',
    genres=['Science Fiction', 'romance'],
    keywords=['space', 'time'],
    top_n=15
)

recommendations


Inputs:
  Movie: Interstellar
  Director: Christopher Nolan
  Genres: ['Science Fiction', 'romance']
  Keywords: ['space', 'time']

Calculated Weights:
  movie: 0.400
  director: 0.200
  genre: 0.200
  keyword: 0.200
  actor: 0.100
------------------------------------------------------------
Returning 15 recommendations


,title,director,genres,keywords,year,score
1580,Gattaca,andrewniccol,"[Thriller, Science Fiction, Mystery, Romance]","[genetics, cheating, paraplegic, suicide attem...",1997.0,0.324552
4848,Vanilla Sky,cameroncrowe,"[Mystery, Romance, Science Fiction]","[regret, jealousy, amnesia, dreams, love of on...",2001.0,0.300000
66470,Tenet,christophernolan,"[Action, Thriller, Science Fiction]","[assassin, espionage, spy, time travel, mumbai...",2020.0,0.300000
11115,The Prestige,christophernolan,"[Drama, Mystery, Science Fiction]","[dying and death, suicide, class society, lond...",2006.0,0.300000
12364,Doctor Who,geoffreysax,"[TV Movie, Adventure, Science Fiction]","[clock, time travel, time, space, millennium, ...",1996.0,0.300000
14888,Mr. Nobody,jacovandormael,"[Science Fiction, Drama, Romance]","[time travel, surrealism, time, choice, free w...",2009.0,0.300000
14844,Inception,christophernolan,"[Action, Science Fiction, Adventure]","[rescue, mission, dreams, airplane, paris, fra...",2010.0,0.300000
43598,Passengers,mortentyldum,"[Drama, Romance, Science Fiction]","[android, spacecraft, asteroid, isolation, sho...",2016.0,0.300000
47952,Nude on the Moon,doriswishman,"[Comedy, Fantasy, Romance, Science Fiction]","[moon, exploitation, space, nudism, woman dire...",1961.0,0.300000
45309,Space Mutiny,nealsundstrom,"[Romance, Adventure, Science Fiction]","[spacecraft, dancing, mutiny, corruption, rock...",1988.0,0.300000


PIPELINE FUNCTION - END IO


In [103]:
import numpy as np

class InteractiveRecommender:
    def __init__(self, movies_df, indices, mat_desc, lookups):
        self.movies = movies_df
        self.indices = indices
        self.mat_desc = mat_desc
        self.lookups = lookups
        self.user_profile = None
        self.profile_keywords = set()

    def build_user_profile(self):
        """
        Step 1: Ingest User Favorites to build a 'Taste Vector'
        """
        print("\n--- STEP 1: TELL ME WHAT YOU LOVE ---")

        # 1. Inputs (Simulated text inputs for notebook)
        print("Enter 3 Favorite Movies (exact titles):")
        fav_movies = [input(f"Movie {i+1}: ").strip() for i in range(3)]

        print("\nEnter 2 Favorite Directors:")
        fav_directors = [input(f"Director {i+1}: ").strip() for i in range(2)]

        print("\nEnter 2 Favorite Genres:")
        fav_genres = [input(f"Genre {i+1}: ").strip() for i in range(2)]

        # 2. Construct the Profile Vector (Aggregation Strategy)
        # We start with a zero vector and add the vectors of all favorites
        profile_vector = np.zeros(self.mat_desc.shape[1]) # TF-IDF shape
        valid_inputs = 0

        # Add Movie Plot Vectors
        for m in fav_movies:
            if m in self.indices:
                idx = self.indices[m]
                # In a real app, we'd use dense vectors. Here we use the TF-IDF row.
                # Note: Sparse matrix addition is slow-ish but fine for 3 items.
                profile_vector += self.mat_desc[idx].toarray()[0]
                valid_inputs += 1

                # Collect keywords for metadata matching
                kws = self.movies.at[idx, 'keywords']
                if isinstance(kws, list):
                    self.profile_keywords.update(kws)

        # We also collect the 'Taste' tags from Directors/Genres for later boosting
        self.profile_tags = {
            'directors': [d.lower().replace(" ", "") for d in fav_directors if d],
            'genres': [g.lower().replace(" ", "") for g in fav_genres if g]
        }

        # Normalize the profile vector
        if valid_inputs > 0:
            profile_vector /= valid_inputs
            self.user_profile = profile_vector
            print("\n✅ Profile Built! I have analyzed your taste.")
        else:
            print("\n⚠️ Could not find those movies. Profile is empty.")

    def get_session_recommendations(self):
        """
        Step 2: Ask for Current Mood and Recommend
        """
        if self.user_profile is None:
            print("Please build a profile first!")
            return

        print("\n--- STEP 2: WHAT ARE YOU WATCHING NOW? ---")
        query_movie = input("Reference Movie (leave blank if none): ").strip()
        query_keywords = input("Keywords/Mood (e.g., 'dark space travel'): ").strip()

        # --- SCORING LOGIC ---

        # 1. Base Score (The Current Query)
        # This is the "King". We calculate this first.
        # We assume 'get_dynamic_recommendations' returns raw scores/indices internally
        # For this demo, we will calculate raw scores manually here

        total_scores = np.zeros(len(self.movies))

        # A. Query Movie Score
        if query_movie and query_movie in self.indices:
            idx = self.indices[query_movie]
            sim_scores = cosine_similarity(self.mat_desc[idx], self.mat_desc).flatten()
            total_scores += sim_scores * 1.0 # High weight for explicit request

        # B. Keyword/Mood Score (TF-IDF transformation of query)
        if query_keywords:
            # Transform user query into vector space
            query_vec = tfidf.transform([query_keywords])
            kw_scores = cosine_similarity(query_vec, self.mat_desc).flatten()
            total_scores += kw_scores * 0.8

        # 2. Personalization Boost (The "Match" Logic)
        # We only apply the profile boost if it makes sense.

        # Calculate alignment between Profile and Current Query
        # (Simplified: Do keywords overlap?)
        query_tokens = set(query_keywords.lower().split())
        overlap = query_tokens.intersection(self.profile_keywords)

        personalization_weight = 0.0
        if len(overlap) > 0 or query_movie:
            print(f"   -> Match found with your profile (Overlap: {overlap})! Personalizing results...")
            personalization_weight = 0.3
        else:
            print("   -> No clear overlap with your history. Prioritizing current query.")
            personalization_weight = 0.05 # Tiny bias just to tie-break

        # Apply Profile Vector similarity
        profile_sim = cosine_similarity(self.user_profile.reshape(1, -1), self.mat_desc).flatten()
        total_scores += profile_sim * personalization_weight

        # Apply Profile Directors/Genres boost (if applicable)
        for d in self.profile_tags['directors']:
            if d in self.lookups['director']:
                # Smaller boost than a direct query match
                total_scores[self.lookups['director'][d]] += 0.2

        # 3. Final Sort & Display
        top_indices = np.argsort(total_scores)[::-1][:25]
        results = self.movies.iloc[top_indices][['title', 'vote_average', 'director', 'genres']]
        results['match_score'] = total_scores[top_indices]

        return results

# --- HOW TO RUN ---
# 1. Initialize
app = InteractiveRecommender(movies, indices, tfidf_matrix, lookups)

# 2. Build Profile (User types inputs)
app.build_user_profile()

# 3. Get Recommendations
recs = app.get_session_recommendations()
print("\n=== YOUR PERSONALIZED PICKS ===")
print(recs)


--- STEP 1: TELL ME WHAT YOU LOVE ---
Enter 3 Favorite Movies (exact titles):
Movie 1: The Dark Knight Rises
Movie 2: Interstellar
Movie 3: A Beautiful Mind

Enter 2 Favorite Directors:
Director 1: Christopher Nolan
Director 2: David Fincher

Enter 2 Favorite Genres:
Genre 1: Science Fiction
Genre 2: Thriller

✅ Profile Built! I have analyzed your taste.

--- STEP 2: WHAT ARE YOU WATCHING NOW? ---
Reference Movie (leave blank if none): Thriller
Keywords/Mood (e.g., 'dark space travel'): Dark
   -> Match found with your profile (Overlap: set())! Personalizing results...

=== YOUR PERSONALIZED PICKS ===
                                     title  vote_average          director  \
59629                             Thriller         4.319     dallasjackson   
17346                The Dark Knight Rises         7.789  christophernolan   
20984                         Interstellar         8.463  christophernolan   
46                                   Se7en         8.377      davidfincher   


In [104]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class InteractiveRecommender:
    def __init__(self, movies_df, indices, mat_desc, lookups):
        self.movies = movies_df
        self.indices = indices
        self.mat_desc = mat_desc
        self.lookups = lookups

        # State
        self.user_profile = None
        self.profile_keywords = set()
        self.profile_tags = {}
        self.fav_indices = set() # NEW: Track what user already knows/likes

    def build_user_profile(self):
        """
        Step 1: Ingest User Favorites to build a 'Taste Vector'
        """
        print("\n--- STEP 1: TELL ME WHAT YOU LOVE ---")

        # 1. Inputs (Simulated)
        print("Enter 3 Favorite Movies (exact titles):")
        fav_movies = [input(f"Movie {i+1}: ").strip() for i in range(3)]

        print("\nEnter 2 Favorite Directors:")
        fav_directors = [input(f"Director {i+1}: ").strip() for i in range(2)]

        print("\nEnter 2 Favorite Genres:")
        fav_genres = [input(f"Genre {i+1}: ").strip() for i in range(2)]

        # 2. Construct the Profile Vector
        profile_vector = np.zeros(self.mat_desc.shape[1])
        valid_inputs = 0

        # Reset favorites list
        self.fav_indices = set()

        for m in fav_movies:
            if m in self.indices:
                idx = self.indices[m]

                # Add to vector
                profile_vector += self.mat_desc[idx].toarray()[0]
                valid_inputs += 1

                # NEW: Add to exclusion list so we don't recommend it back
                self.fav_indices.add(idx)

                # Collect keywords
                kws = self.movies.at[idx, 'keywords']
                if isinstance(kws, list):
                    self.profile_keywords.update(kws)

        self.profile_tags = {
            'directors': [d.lower().replace(" ", "") for d in fav_directors if d],
            'genres': [g.lower().replace(" ", "") for g in fav_genres if g]
        }

        if valid_inputs > 0:
            profile_vector /= valid_inputs
            self.user_profile = profile_vector
            print(f"\n✅ Profile Built based on {valid_inputs} movies.")
        else:
            print("\n⚠️ Could not find those movies. Profile is empty.")

    def get_session_recommendations(self):
        """
        Step 2: Ask for Current Mood and Recommend (Filtering Favorites)
        """
        if self.user_profile is None:
            print("Please build a profile first!")
            return

        print("\n--- STEP 2: WHAT ARE YOU WATCHING NOW? ---")
        query_movie = input("Reference Movie (leave blank if none): ").strip()
        query_keywords = input("Keywords/Mood (e.g., 'dark space travel'): ").strip()

        # --- SCORING LOGIC ---
        total_scores = np.zeros(len(self.movies))

        # Track exclusions for this specific search
        current_exclusions = self.fav_indices.copy()

        # 1. Base Score (The Current Query)
        if query_movie and query_movie in self.indices:
            idx = self.indices[query_movie]
            sim_scores = cosine_similarity(self.mat_desc[idx], self.mat_desc).flatten()
            total_scores += sim_scores * 1.0
            current_exclusions.add(idx) # Don't recommend the reference movie itself

        if query_keywords:
            query_vec = tfidf.transform([query_keywords])
            kw_scores = cosine_similarity(query_vec, self.mat_desc).flatten()
            total_scores += kw_scores * 0.8

        # 2. Personalization Boost
        query_tokens = set(query_keywords.lower().split())
        overlap = query_tokens.intersection(self.profile_keywords)

        personalization_weight = 0.0
        if len(overlap) > 0 or query_movie:
            print(f"   -> Match found with profile! Personalizing...")
            personalization_weight = 0.3
        else:
            print("   -> Exploration Mode (Low profile bias).")
            personalization_weight = 0.05

        # Apply Profile Vector similarity
        profile_sim = cosine_similarity(self.user_profile.reshape(1, -1), self.mat_desc).flatten()
        total_scores += profile_sim * personalization_weight

        # Apply Profile Directors/Genres boost
        for d in self.profile_tags['directors']:
            if d in self.lookups['director']:
                total_scores[self.lookups['director'][d]] += 0.2

        # 3. Final Sort & Filter
        # Sort ALL movies first
        top_indices = np.argsort(total_scores)[::-1]

        # Filter out exclusions (Favorites + Current Seed)
        final_indices = [i for i in top_indices if i not in current_exclusions]

        # NEW: Return top 25 instead of 10
        top_25_indices = final_indices[:25]

        results = self.movies.iloc[top_25_indices][['title', 'vote_average', 'director', 'genres']]
        results['match_score'] = total_scores[top_25_indices]

        return results

# --- RUNNER ---
app = InteractiveRecommender(movies, indices, tfidf_matrix, lookups)
app.build_user_profile()
recs = app.get_session_recommendations()
print("\n=== TOP 25 PERSONALIZED PICKS ===")
print(recs)


--- STEP 1: TELL ME WHAT YOU LOVE ---
Enter 3 Favorite Movies (exact titles):
Movie 1: Interstellar
Movie 2: The dark knight rises
Movie 3: A beautiful Mind

Enter 2 Favorite Directors:
Director 1: Christopher Nolan
Director 2: David fincher

Enter 2 Favorite Genres:
Genre 1: thriller
Genre 2: comedy

✅ Profile Built based on 1 movies.

--- STEP 2: WHAT ARE YOU WATCHING NOW? ---
Reference Movie (leave blank if none): Happy Death Day
Keywords/Mood (e.g., 'dark space travel'): Dark
   -> Match found with profile! Personalizing...

=== TOP 25 PERSONALIZED PICKS ===
                                     title  vote_average           director  \
46                                   Se7en         8.377       davidfincher   
17346                The Dark Knight Rises         7.789   christophernolan   
66470                                Tenet         7.176   christophernolan   
5258                              Insomnia         6.956   christophernolan   
11385                              